**Context**: 
This is a transnational data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.The company mainly sells unique all-occasion gifts. Many customers of the company are wholesalers.

**Data Description**:

InvoiceNo: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.

StockCode: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.

Description: Product (item) name. Nominal. 

Quantity: The quantities of each product (item) per transaction. Numeric. 

InvoiceDate: Invoice Date and time. Numeric, the day and time when each transaction was generated.

UnitPrice: Unit price. Numeric, Product price per unit in sterling. 

CustomerID: Customer number. Nominal, a 5-digit integral number uniquely assigned to each customer.

Country: Country name. Nominal, the name of the country where each customer resides.

**Problem statement**:

It is a business critical requirement to understand the value derived from a customer. RFM is a method used for analyzing customer value.
Perform customer segmentation using RFM analysis. The resulting segments can be ordered from most valuable (highest recency, frequency, and value) to least valuable (lowest recency, frequency, and value). Identifying the most valuable RFM segments can capitalize on chance relationships in the data used for this analysis.

**Approach**:

Following pointers will be helpful to structure your findings.   


**1.	Perform a preliminary data inspection and Data cleaning**

a.	Check for missing data and formulate apt strategy to treat them.
b.	Are there any duplicate data records? Remove them if present.
c.	Perform Descriptive analytics on the given data.

**2.	Cohort Analysis: A cohort is a group of subjects who share a defining characteristic. We can observe how a cohort behaves across time and compare it to other cohorts.**

a.	Create month cohorts and analyse active  customers for each cohort.
b.	Also Analyse the retention rate of customers. Comment.

**3.	Build a RFM model – Recency Frequency and Monetary based on their behaviour.**

**Recency** is about when was the last order of a customer. It means the number of days since a customer made the last purchase. If it’s a case for a website or an app, this could be interpreted as the last visit day or the last login time.
**Frequency** is about the number of purchase in a given period. It could be 3 months, 6 months or 1 year. So we can understand this value as for how often or how many a customer used the product of a company. The bigger the value is, the more engaged the customers are. Could we say them as our VIP? Not necessary. Cause we also have to think about how much they actually paid for each purchase, which means monetary value.
**Monetary** is the total amount of money a customer spent in that given period. Therefore big spenders will be differentiated with other customers such as MVP or VIP.

a.	Calculate RFM metrics.

i.	Recency as the time in no. of days since last transaction
ii.	Frequency as  count of purchases done 
iii.	Monetary value  as total amount spend 

b.	Build RFM Segments.

i.	Give Recency Frequency and Monetary scores individually by dividing them in to quartiles.
Note: Rate "Recency" for customer who have been active more recently better than the less recent customer, because each company wants its customers to be recent 
Rate "Frequency" and "Monetary Value" higher label because we want Customer to spend more money and visit more often.
ii.	Combine three ratings to get a RFM segment (as strings)
iii.	Get the RFM score by adding up the three ratings.

c.	Analyse the RFM Segments by summarizing them and comment on the findings.

**4.	Create clusters using k means clustering algorithm.**

a.	Prepare the data for the algorithm.

i.	If the data is Un Symmetrically distributed, manage the skewness with appropriate transformation.
ii.	Standardize / scale the data.

b.	Decide the optimum number of clusters to be formed

c.	Analyse these clusters and comment on the results.


In [ ]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df_train=pd.read_excel("train.xlsx")
df_test=pd.read_excel("test.xlsx")

In [ ]:
df_train.head()

In [ ]:
print(df_train.shape,df_train.size)
print(df_test.shape,df_test.size)


In [ ]:
df_test.head()

In [ ]:
df_train.isna().sum()

In [ ]:
df_test.isnull().sum()

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
# Calculating the Missing Values % contribution in DF
df_null = round(df_train.isnull().sum()/len(df_train)*100,2)
df_null

As we can see two columns in data have missing values.

Description - 0.25% (963 nos.)
CustomerID - 24.85% (94260)

CustomerID is important feature of our analysis since our analysis is centered around Customers only so we can not impute null values CustomerID with mean/ median/ mode in this case. We will check possibility to fill null values in CustomerID column by looking up for InvoiceNo of the row having null CustomerID in other rows where CustomerID is present. If there are still any null values in CustomerID after this process then we will drop complete row having missing CustomerID.

We can drop **Description** feature from our data since it is not not going to contribute in our model.

In [ ]:
#for i in df_train['CustomerID']:
    #if i=='NaN':
        #invoice_null_custid_test = set(df_test[df_test['CustomerID'].isnull()]['InvoiceNo'])
        #df_test[df_test['InvoiceNo'].isin(invoice_null_custid_test) & (~df_test['CustomerID'].isnull())]

In [ ]:
df_train.isnull().sum()

In [ ]:
# Replace null values in CustomerID column with corresponding InvoiceNo value
df_train['CustomerID'] = df_train.apply(lambda x: x['InvoiceNo'] if pd.isnull(x['CustomerID']) else x['CustomerID'], axis=1)

# or
df_test['CustomerID'] = df_test.apply(lambda x: x['InvoiceNo'] if pd.isnull(x['CustomerID']) else x['CustomerID'], axis=1)

#data['CustomerID'].fillna(data['InvoiceNo'], inplace=True)


In [ ]:
df_train.isnull().sum()

In [ ]:
df_test.isnull().sum()

In [ ]:
df_train=df_train.drop('Description',axis=1)
df_test=df_test.drop('Description',axis=1)

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
df_train.isnull().sum()

In [ ]:
df_train.info()

### **Remove duplicate data records** :  
Since our data is transactional data and it has duplicate entries for InvoiceNo and CustomerID, we will drop only those rows which are completely duplicated, not on the basis of any one particular column such as InvoiceNo or CustomerID etc.

In [ ]:
df_train= df_train.drop_duplicates()
df_test=df_test.drop_duplicates()
print(df_train.shape,df_test.shape)

In [ ]:
df_train['CustomerID'] = df_train['CustomerID'].astype(str)
df_test['CustomerID'] = df_test['CustomerID'].astype(str)


In [ ]:
df_train.describe(datetime_is_numeric=True)

**Quantity**: Average quantity of each product in transaction is 9.56. Also note that minimum value in Quantity column is negative. This implies that some customers had returned the product during our period of analysis.

**InvoiceDate**: Our data has transaction between 01-12-2010 to 09-12-2011

**UnitPrice**: Average price of each product in transactions is 4.697

### Data Transformation
#### Perform Cohort Analysis

Create month cohort of customers and analyze active customers in each cohort:

In [ ]:
# Convert to InvoiceDate to Year-Month format
df_train['month_year'] = df_train['InvoiceDate'].dt.to_period('M')
df_train['month_year'].nunique()

In [ ]:
month_cohort = df_train.groupby('month_year')['CustomerID'].nunique()
month_cohort

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(y = month_cohort.index, x = month_cohort.values);
plt.xlabel("Count of customers")
plt.title("No. of active customers in each month")

In [ ]:
month_cohort - month_cohort.shift(1)

In [ ]:
retention_rate = round(month_cohort.pct_change(periods=1)*100,2)
retention_rate

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(y = retention_rate.index, x = retention_rate.values);
plt.xlabel("Retention (in %)")
plt.title("Month-wise customer retention rate");

#### Monetary analysis:

In [ ]:
df_train['amount'] = df_train['Quantity']*df_train['UnitPrice']
df_train.head()

In [ ]:
df_monetary = df_train.groupby('CustomerID').sum()['amount'].reset_index()
df_monetary

#### Frequency Analysis

In [ ]:
df_frequency = df_train.groupby('CustomerID').nunique()['InvoiceNo'].reset_index()
# df_freqency = df.drop_duplicates('InvoiceNo').groupby('CustomerID').count()['InvoiceNo'].reset_index()
df_frequency

#### Recency Analysis

In [ ]:
# We will fix reference date for calculating recency as last transaction day in data + 1 day
from datetime import timedelta
ref_day = max(df_train['InvoiceDate']) + timedelta(days=1)
df_train['days_to_last_order'] = (ref_day - df_train['InvoiceDate']).dt.days
df_train.head()

In [ ]:
df_recency = df_train.groupby('CustomerID')['days_to_last_order'].min().reset_index()
df_recency

#### Calculate RFM metrics

In [ ]:
df_rf = pd.merge(df_recency, df_frequency,  on='CustomerID', how='inner')
df_rfm = pd.merge(df_rf, df_monetary, on='CustomerID', how='inner')
df_rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
df_rfm.head()

In [ ]:
#Build RFM Segments:
df_rfm['recency_labels'] = pd.cut(df_rfm['Recency'], bins=5,
                                     labels=['newest', 'newer', 'medium', 'older', 'oldest'])
df_rfm['recency_labels'].value_counts().plot(kind='barh');
df_rfm['recency_labels'].value_counts()

In [ ]:
df_rfm['frequency_labels'] = pd.cut(df_rfm['Frequency'], bins=5, labels=['lowest', 'lower', 'medium', 'higher', 'highest'])
df_rfm['frequency_labels'].value_counts().plot(kind='barh');
df_rfm['frequency_labels'].value_counts()

In [ ]:
df_rfm['monetary_labels'] = pd.cut(df_rfm['Monetary'], bins=5, labels=['smallest', 'smaller', 'medium', 'larger', 'largest'])
df_rfm['monetary_labels'].value_counts().plot(kind='barh');
df_rfm['monetary_labels'].value_counts()

In [ ]:
df_rfm['rfm_segment'] = df_rfm[['recency_labels','frequency_labels','monetary_labels']].agg('-'.join, axis=1)
df_rfm.head()

In [ ]:
#RFM score
recency_dict = {'newest': 5, 'newer':4, 'medium': 3, 'older':2, 'oldest':1}
frequency_dict = {'lowest':1, 'lower':2, 'medium': 3, 'higher':4, 'highest':5}
monetary_dict = {'smallest':1, 'smaller':2, 'medium': 3, 'larger':4, 'largest':5}

df_rfm['rfm_score'] = df_rfm['recency_labels'].map(recency_dict).astype(int)+ df_rfm['frequency_labels'].map(frequency_dict).astype(int) + df_rfm['monetary_labels'].map(monetary_dict).astype(int)
df_rfm.head(10)

In [ ]:
df_rfm['rfm_segment'].value_counts().plot(kind='barh', figsize=(10, 5));


In [ ]:
df_rfm['rfm_score'].value_counts().plot(kind='barh', figsize=(10, 5));


Week 3
Data Modeling:
Create clusters using k-means clustering algorithm.

a. Prepare the data for the algorithm. If the data is asymmetrically distributed, manage the skewness with appropriate transformation. Standardize the data.

In [ ]:
print(df_rfm.shape)
df_rfm.head()

In [ ]:
plt.figure(figsize=(12,6))

for i, feature in enumerate(['Recency', 'Frequency', 'Monetary']):
    plt.subplot(2,3,i+1)
    df_rfm[feature].plot(kind='box')
    plt.subplot(2,3,i+1+3)
    df_rfm[feature].plot(kind='hist')

In [ ]:
df_rfm = df_rfm[(df_rfm['Frequency']<60) & (df_rfm['Monetary']<40000)]
df_rfm.shape

In [ ]:
plt.figure(figsize=(12,6))

for i, feature in enumerate(['Recency', 'Frequency', 'Monetary']):
    plt.subplot(2,3,i+1)
    df_rfm[feature].plot(kind='box')
    plt.subplot(2,3,i+1+3)
    df_rfm[feature].plot(kind='hist')

In [ ]:
df_rfm_log_trans = pd.DataFrame()
df_rfm_log_trans['Recency'] = np.log(df_rfm['Recency'])
df_rfm_log_trans['Frequency'] = np.log(df_rfm['Frequency'])
df_rfm_log_trans['Monetary'] = np.log(df_rfm['Monetary']-df_rfm['Monetary'].min()+1)

In [ ]:
#Standard Scalar Transformation: It is extremely important to rescale the features so that they have a comparable scale.
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df_rfm_scaled = scaler.fit_transform(df_rfm_log_trans[['Recency', 'Frequency', 'Monetary']])
df_rfm_scaled

df_rfm_scaled = pd.DataFrame(df_rfm_scaled)
df_rfm_scaled.columns = ['Recency', 'Frequency', 'Monetary']
df_rfm_scaled.head()

#### b. Build K-Means Clustering Model and Decide the optimum number of clusters to be formed.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import timedelta
from pandas import ExcelWriter

In [ ]:
# k-means with some arbitrary k
kmeans = KMeans(n_clusters=3, max_iter=50)
kmeans.fit(df_rfm_scaled)

In [ ]:
# Finding the Optimal Number of Clusters with the help of Elbow Curve/ SSD
ssd = []
range_n_clusters = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
for num_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=num_clusters, max_iter=100)
    kmeans.fit(df_rfm_scaled)
    
    ssd.append(kmeans.inertia_)
    
# plot the SSDs for each n_clusters
plt.plot(range_n_clusters,ssd);

In [ ]:
# Creating dataframe for exporting to create visualization in tableau later
df_inertia = pd.DataFrame(list(zip(range_n_clusters, ssd)), columns=['clusters', 'intertia'])
df_inertia

In [ ]:
# Finding the Optimal Number of Clusters with the help of Silhouette Analysis
range_n_clusters = [2, 3, 4, 5, 6, 7, 8, 9, 10]

for num_clusters in range_n_clusters:    
    kmeans = KMeans(n_clusters=num_clusters, max_iter=50)
    kmeans.fit(df_rfm_scaled)
    
    cluster_labels = kmeans.labels_
    
    silhouette_avg = silhouette_score(df_rfm_scaled, cluster_labels)
    print("For n_clusters={0}, the silhouette score is {1}".format(num_clusters, silhouette_avg))

In [ ]:
#We can select optimum number of clusters as 3 in our final model
# Final model with k=3
kmeans = KMeans(n_clusters=3, max_iter=50)
kmeans.fit(df_rfm_scaled)

#### c. Analyze these clusters and comment on the results.

In [ ]:
# assign the label
df_rfm['Cluster_Id'] = kmeans.labels_
df_rfm.head()

In [ ]:
# Box plot to visualize Cluster Id vs Monetary
sns.boxplot(x='Cluster_Id', y='Monetary', data=df_rfm);

In [ ]:
# Box plot to visualize Cluster Id vs Frequency
sns.boxplot(x='Cluster_Id', y='Frequency', data=df_rfm);

In [ ]:
# Box plot to visualize Cluster Id vs Recency
sns.boxplot(x='Cluster_Id', y='Recency', data=df_rfm);

**Inference:**

As we can observe from above boxplots that our model has nicely created 3 segements of customer with the interpretation as below:

Customers with Cluster Id 0 are less frequent buyers with low monetary expenditure and also they have not purchased anything in recent time and hence least important for business.

Customers with Cluster Id 1 are the customers having Recency, Frequency and Monetary score in the medium range.

Customers with Cluster Id 2 are the most frequent buyers, spending high amount and recently placing orders so they are the most important customers from business point of view.